In [4]:
import pandas as pd

# Lê apenas as primeiras 10.000 linhas para não travar
file_path = "OBCI_F3.TXT"

# Ignora cabeçalho comentado com "%"
with open(file_path, "r") as f:
    lines = [line for line in f if not line.startswith('%')]

# Lê os primeiros 10.000 (ou menos)
preview_lines = lines[:10000]

# Quebra por vírgula
data = [line.strip().split(",") for line in preview_lines]

# Tenta converter a última coluna (timestamp em segundos tipo 1.75e9)
timestamps = []
for row in data:
    try:
        ts = float(row[-2])  # penúltima coluna = timestamp numérico
        timestamps.append(ts)
    except:
        continue

# Verifica quantos foram lidos
print(f"{len(timestamps)} timestamps lidos.")

# Calcula a frequência de amostragem
import numpy as np
diffs = np.diff(timestamps)
mean_diff = np.mean(diffs)
sampling_rate = 1 / mean_diff

print(f"Frequência estimada: {sampling_rate:.2f} Hz")


448 timestamps lidos.
Frequência estimada: nan Hz


/opt/homebrew/Caskroom/miniforge/base/envs/creativity-eeg/lib/python3.10/site-packages/numpy/lib/function_base.py:1452: RuntimeWarning: invalid value encountered in subtract
  a = op(a[slice1], a[slice2])
/opt/homebrew/Caskroom/miniforge/base/envs/creativity-eeg/lib/python3.10/site-packages/numpy/core/_methods.py:118: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


In [5]:
# Printar algumas linhas pra inspeção
for row in data[:3]:
    print(row)


['0000C9E6']
['00', '9C4933', '800000', '800000', '800000', '94EA1A', '800000', '8042CD', '92433B', '816CE3', '87AA43', '800000', '800000', '857263', '800000', '858592', '800000', '0110', '0020', '1010']
['01', '9C4A34', '800000', '800000', '800000', '94EB0E', '800000', '80436D', '92442B', '816DBD', '87AB6D', '800000', '800000', '857387', '800000', '8586CF', '800000']


In [7]:
with open("OBCI_F3.TXT", "r") as f:
    lines = [line for line in f if not line.startswith('%')]

print(f"Total de amostras: {len(lines)}")


Total de amostras: 2783


In [9]:
import pandas as pd

# Caminho para o seu arquivo CSV (altere conforme necessário)
csv_path = "BrainFlow-RAW_2025-06-21_12-03-26_0.csv"

# Ler o arquivo e tentar detectar separador (pode ser vírgula ou tabulação)
try:
    df = pd.read_csv(csv_path, sep="\t", header=0)
except Exception:
    df = pd.read_csv(csv_path, sep="\t")

# Verificar os nomes das colunas para encontrar o timestamp
print("Colunas detectadas:")
print(df.columns)

# Tentar encontrar a coluna de timestamp automaticamente
timestamp_col = None
for col in df.columns:
    if "timestamp" in col.lower() or "time" in col.lower():
        timestamp_col = col
        break

if timestamp_col is None:
    # Tentar como penúltima coluna (frequente nos CSVs do OpenBCI)
    timestamp_col = df.columns[-2]
    print(f"Nenhuma coluna com nome de timestamp encontrada. Usando: '{timestamp_col}'")

# Converter timestamp para float (Unix time em segundos)
df[timestamp_col] = df[timestamp_col].astype(float)

# Cálculo da duração total e número de amostras
start_time = df[timestamp_col].iloc[0]
end_time = df[timestamp_col].iloc[-1]
duration_sec = end_time - start_time
num_samples = len(df)

# Cálculo da taxa de amostragem
sampling_rate = num_samples / duration_sec

print(f"\n📊 Resultados:")
print(f" - Número de amostras: {num_samples}")
print(f" - Duração total (s): {duration_sec:.6f}")
print(f" - Taxa de amostragem estimada: {sampling_rate:.2f} Hz")


Colunas detectadas:
Index(['0.000000', '-146062.659748', '-187500.022352', '-187500.022352.1',
       '-187500.022352.2', '-156860.876901', '-187500.022352.3',
       '-187115.997030', '-160745.453625', '-92706.050599', '-88135.923521',
       '-93750.011176', '-93750.011176.1', '-89760.627122', '-93750.011176.2',
       '-89705.753589', '-93750.011176.3', '0.034000', '0.004000', '0.514000',
       '192.000000', '0.500000', '8.000000', '0.000000.1', '16.000000',
       '8.000000.1', '8.000000.2', '0.000000.2', '0.000000.3', '0.000000.4',
       '1750500245.678244', '0.000000.5'],
      dtype='object')
Nenhuma coluna com nome de timestamp encontrada. Usando: '1750500245.678244'

📊 Resultados:
 - Número de amostras: 1379
 - Duração total (s): 11.803886
 - Taxa de amostragem estimada: 116.83 Hz


In [11]:
import pandas as pd

# Caminho para seu arquivo CSV
csv_path = "BrainFlow-RAW_2025-06-21_12-03-26_0.csv"

# Ler com tab como separador (separação por vírgula ou tab)
try:
    df = pd.read_csv(csv_path, sep="\t", header=0)
except:
    df = pd.read_csv(csv_path, sep="\t", engine="python")

# Exibir as colunas para você confirmar
print("Colunas encontradas:", df.columns.tolist())

# Encontrar o índice da coluna de timestamp Unix
# Consideramos penúltima coluna
timestamp_col = df.columns[-2]
print("Usando timestamp em:", timestamp_col)

# Converter para float
df[timestamp_col] = df[timestamp_col].astype(float)

# Pegamos o tempo do início e fim
t0 = df[timestamp_col].iloc[0]
tN = df[timestamp_col].iloc[-1]
duration = tN - t0

# Número total de amostras
N = len(df)

# A taxa de amostragem estimada
srate = N / duration

print(f"Duração total: {duration:.4f} s")
print(f"Número de amostras: {N}")
print(f"Taxa de amostragem estimada: {srate:.2f} Hz")


Colunas encontradas: ['0.000000', '-146062.659748', '-187500.022352', '-187500.022352.1', '-187500.022352.2', '-156860.876901', '-187500.022352.3', '-187115.997030', '-160745.453625', '-92706.050599', '-88135.923521', '-93750.011176', '-93750.011176.1', '-89760.627122', '-93750.011176.2', '-89705.753589', '-93750.011176.3', '0.034000', '0.004000', '0.514000', '192.000000', '0.500000', '8.000000', '0.000000.1', '16.000000', '8.000000.1', '8.000000.2', '0.000000.2', '0.000000.3', '0.000000.4', '1750500245.678244', '0.000000.5']
Usando timestamp em: 1750500245.678244
Duração total: 11.8039 s
Número de amostras: 1379
Taxa de amostragem estimada: 116.83 Hz


In [12]:
with open("OBCI_F3.TXT") as f:
    lines = [l for l in f if not l.startswith('%')]
print("Linhas:", len(lines))


Linhas: 2783


In [13]:
size_est = len(lines) * 200 / 1024 / 1024  # em MB
print(f"Estimado: {size_est:.2f} MB")


Estimado: 0.53 MB
